In [1]:
# install Roboflow and helpers
%pip install roboflow python-dotenv

from roboflow import Roboflow
from dotenv import load_dotenv
from google.colab import userdata
import os

# load environment variables from .env (optional, if you use one)
load_dotenv()

# in Colab, store your API key in the "Secrets" sidebar as ROBOFLOW_API_KEY
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
api_key = os.getenv("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=api_key)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
# example: download a dataset from a Roboflow workspace
WORKSPACE = "caretech-qnw0y"                # your Roboflow workspace slug
PROJECT   = "v1-caretech-combined-dataset"  # your project slug
VERSION   = 1                               # dataset version number
FORMAT    = "yolov8"                        # export format (e.g. "yolov5", "coco", "voc")

project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download(FORMAT)

print("Dataset location:", dataset.location)
DATASET_PATH = dataset.location

# verify amount of images
print('Training:', len(os.listdir(os.path.join(DATASET_PATH, "train/images"))))
print('Testing:', len(os.listdir(os.path.join(DATASET_PATH, "test/images"))))
print('Valid:', len(os.listdir(os.path.join(DATASET_PATH, "valid/images"))))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to [V1]-CareTech-Combined-Dataset-1 in yolov8:: 100%|██████████| 58572/58572 [00:17<00:00, 3384.54it/s]


Dataset location: /content/[V1]-CareTech-Combined-Dataset-1
Training: 20509
Testing: 2949
Valid: 5822


In [10]:
import yaml, os

DATASET = "/content/[V1]-CareTech-Combined-Dataset-1"
YAML_PATH = f"{DATASET}/data.yaml"

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

# works for both list or dict style yaml
if isinstance(data["names"], dict):
    id_to_name = {int(k):v for k,v in data["names"].items()}
else:
    id_to_name = {i:v for i,v in enumerate(data["names"])}

name_to_id = {v:k for k,v in id_to_name.items()}

print("Loaded classes:", len(id_to_name))


Loaded classes: 258


In [11]:
MERGE_RULES = {

    # ALL noodle soups
    "noodle_soup": [
        "ramen-noodle","tanmen","udon-noodle","soba-noodle","tempura-udon",
        "khao-soi","laksa","pho","beef-noodle-soup","champon",
        "hue-beef-rice-vermicelli-soup","tensin-noodle"
    ],

    # noodle stir fry
    "fried_noodles": [
        "chow-mein","fried-noodle","mie-goreng","yakisoba",
        "crispy-noodles","dipping-noodles"
    ],

    # rice bowls / rice plates
    "rice_bowl": [
        "pork-cutlet-on-rice","chicken-n-egg-on-rice",
        "minced-pork-rice","beef-bowl","broiled-eel-bowl"
    ],

    # curries
    "curry": [
        "green-curry","yellow-curry","dry-curry",
        "beef-curry","cutlet-curry","chicken-rice-curry-with-coconut"
    ],

    # sushi family
    "sushi": [
        "sushi","sushi-bowl","inarizushi","sashimi-bowl"
    ],

    # dumplings
    "dumplings": [
        "jiaozi","xiao-long-bao","steamed-meat-dumpling",
        "fried-pork-dumplings-served-in-soup"
    ],

    # pork cutlets
    "cutlet": [
        "pork-cutlet","pork-loin-cutlet","pork-fillet-cutlet",
        "sirloin-cutlet","ham-cutlet","chicken-cutlet"
    ],

    # salads
    "salad": [
        "caesar-salad","green-salad","potato-salad","macaroni-salad"
    ],

    # soups
    "soup": [
        "miso-soup","clear-soup","hot-and-sour-soup",
        "winter-melon-soup","potage","minestrone","oxtail-soup"
    ],

    # desserts
    "dessert": [
        "shortcake","rare-cheese-cake","tiramisu","parfait",
        "custard-tart","cream-puff","mango-pudding"
    ]
}


In [16]:
import shutil, os

MERGED_PATH = "/content/caretech_merged_dataset"

if os.path.exists(MERGED_PATH):
    shutil.rmtree(MERGED_PATH)

shutil.copytree(DATASET_PATH, MERGED_PATH)

print("Working on:", MERGED_PATH)


Working on: /content/caretech_merged_dataset


In [14]:
old_to_new = {}
new_names = []
used_old_ids = set()

# create merged classes
for new_class, members in MERGE_RULES.items():
    new_id = len(new_names)
    new_names.append(new_class)

    for m in members:
        if m in name_to_id:
            old_id = name_to_id[m]
            old_to_new[old_id] = new_id
            used_old_ids.add(old_id)

# keep all other classes unchanged
for old_id, name in id_to_name.items():
    if old_id not in used_old_ids:
        new_id = len(new_names)
        new_names.append(name)
        old_to_new[old_id] = new_id

print("Old classes:", len(id_to_name))
print("New classes:", len(new_names))

print("\nMapping:")
for k in list(old_to_new.keys()):
    print(id_to_name[k], "→", new_names[old_to_new[k]])



Old classes: 258
New classes: 208

Mapping:
ramen-noodle → noodle_soup
tanmen → noodle_soup
udon-noodle → noodle_soup
soba-noodle → noodle_soup
tempura-udon → noodle_soup
khao-soi → noodle_soup
laksa → noodle_soup
pho → noodle_soup
beef-noodle-soup → noodle_soup
champon → noodle_soup
hue-beef-rice-vermicelli-soup → noodle_soup
tensin-noodle → noodle_soup
chow-mein → fried_noodles
fried-noodle → fried_noodles
mie-goreng → fried_noodles
crispy-noodles → fried_noodles
dipping-noodles → fried_noodles
pork-cutlet-on-rice → rice_bowl
chicken-n-egg-on-rice → rice_bowl
minced-pork-rice → rice_bowl
beef-bowl → rice_bowl
broiled-eel-bowl → rice_bowl
green-curry → curry
yellow-curry → curry
dry-curry → curry
beef-curry → curry
cutlet-curry → curry
chicken-rice-curry-with-coconut → curry
sushi → sushi
sushi-bowl → sushi
inarizushi → sushi
sashimi-bowl → sushi
jiaozi → dumplings
xiao-long-bao → dumplings
steamed-meat-dumpling → dumplings
fried-pork-dumplings-served-in-soup → dumplings
pork-cutlet →

In [17]:
def relabel_split(split):

    labels_dir = f"{MERGED_PATH}/{split}/labels"
    images_dir = f"{MERGED_PATH}/{split}/images"

    removed = 0

    for file in os.listdir(labels_dir):

        label_path = os.path.join(labels_dir,file)

        with open(label_path) as f:
            lines=f.readlines()

        new_lines=[]

        for line in lines:
            parts=line.strip().split()
            old=int(parts[0])

            # convert class id
            parts[0]=str(old_to_new[old])
            new_lines.append(" ".join(parts))

        if len(new_lines)==0:
            os.remove(label_path)

            img=os.path.join(images_dir,file.replace(".txt",".jpg"))
            if os.path.exists(img):
                os.remove(img)
            removed+=1

        else:
            with open(label_path,"w") as f:
                f.write("\n".join(new_lines))

    print(split,"removed:",removed)


for split in ["train","valid","test"]:
    relabel_split(split)


train removed: 0
valid removed: 0
test removed: 0


In [18]:
import yaml

new_yaml = {
    "path": MERGED_PATH,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(new_names),
    "names": new_names
}

yaml_path = f"{MERGED_PATH}/data.yaml"

with open(yaml_path,"w") as f:
    yaml.dump(new_yaml,f)

print("New class count:",len(new_names))
print("YAML saved to:",yaml_path)


New class count: 208
YAML saved to: /content/caretech_merged_dataset/data.yaml


In [20]:
import roboflow
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)

WORKSPACE_SLUG = "caretechnathanrushil"
workspace = rf.workspace(WORKSPACE_SLUG)

workspace.upload_dataset(
   MERGED_PATH,
   "class-merging",
   num_workers=10,
   project_license="MIT",
   project_type="object-detection",
   batch_name=None,
   num_retries=0,
   is_prediction=False,
)

Streaming output truncated to the last 5000 lines.
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_326732_jpg.rf.9d5b4c48606d55e2e7c3fc5fe5452398.jpg (NNi8uDVoo4QVwgER53G1) [0.4s] / annotations = OK [0.4s]
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_326881_jpg.rf.f9f633f71ce80592e1e4e447dffe1eba.jpg (9s0GGADcCBxCxDiAsqDD) [0.4s] / annotations = OK [0.4s]
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_325984_jpg.rf.7c84ad6a561f3b4e3261190c32cbeb74.jpg (YCUy9X8xjenJzCcjDrTy) [0.6s] / annotations = OK [0.5s]
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_327316_jpg.rf.2291ff9c41dbb7279cdf70e138e30cdf.jpg (QudLbb7Ywy6y54HbxKlS) [0.5s] / annotations = OK [0.4s]
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_327705_jpg.rf.77bc984a63220db4ed402525c151886c.jpg (ywFakOvZnGdGeXIhf31k) [0.5s] / annotations = OK [0.3s]
[UPLOADED] /content/caretech_merged_dataset/valid/images/chow_mein_327752_jpg.r